In [ ]:
!pip install -q "transformers<5" sentence-transformers faiss-cpu torch accelerate gradio pypdf python-docx

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/mariamessam47/legal-contracts-eg")
CONTRACTS_DIR = DATA_DIR
LAWS_DIR = DATA_DIR
LABELS_PATH = DATA_DIR / "risk_labels.json"

print(os.listdir(DATA_DIR) if DATA_DIR.exists() else "المسار غير موجود")

In [ ]:
CONTRACT_FILE_PREFIXES = ("rent_", "employment_", "supply_")

def load_contracts(contracts_dir):
    contracts = {}
    for file in sorted(Path(contracts_dir).glob("*.txt")):
        if file.name.startswith(CONTRACT_FILE_PREFIXES):
            with open(file, "r", encoding="utf-8") as f:
                contracts[file.stem] = f.read()
    return contracts

contracts = load_contracts(CONTRACTS_DIR)
print(f"تم تحميل {len(contracts)} عقد:", list(contracts.keys()))

In [ ]:
def split_into_clauses(contract_text):
    parts = re.split(r"(البند\s+\S+)", contract_text)
    clauses = []
    current_title = None
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if part.startswith("البند"):
            current_title = part
        elif current_title:
            clauses.append({"title": current_title, "text": part})
            current_title = None
    return clauses

In [ ]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

CONTRACT_LABELS = {
    "rental": "عقد إيجار",
    "employment": "عقد عمل",
    "supply": "عقد توريد بضائع"
}

# كلمات مفتاحية مميزة لكل نوع عقد (Domain Keywords) - أدق بكتير من Zero-shot العام في نصوص قانونية
CONTRACT_KEYWORDS = {
    "rental": ["مؤجر", "مستأجر", "الأجرة", "إيجار", "العين المؤجرة", "الإخلاء", "التأمين النقدي", "الوحدة", "المحل", "الشقة"],
    "employment": ["صاحب العمل", "العامل", "الموظف", "الأجر الشهري", "فترة الاختبار", "الإجازة", "ساعات العمل", "مكافأة نهاية الخدمة", "الفصل"],
    "supply": ["المورد", "التوريد", "الشحنة", "المشتري", "البضاعة", "المطابقة", "غرامة التأخير", "الفحص"]
}

def classify_contract_type(text):
    """نظام هجين: نحسب أولًا تكرار الكلمات المفتاحية لكل نوع، ولو كانت النتيجة واضحة كفاية نعتمدها،
    وإلا نلجأ لـ Zero-shot classification كخطة بديلة."""

    keyword_scores = {}
    for contract_key, keywords in CONTRACT_KEYWORDS.items():
        count = sum(text.count(kw) for kw in keywords)
        keyword_scores[contract_key] = count

    best_key = max(keyword_scores, key=keyword_scores.get)
    best_count = keyword_scores[best_key]
    total_count = sum(keyword_scores.values())

    # لو النتيجة بالكلمات المفتاحية واضحة (فارق واضح عن باقي الأنواع) نعتمدها مباشرة بثقة عالية
    if total_count > 0 and best_count / total_count >= 0.5:
        confidence = min(0.95, 0.6 + (best_count / total_count) * 0.4)
        return CONTRACT_LABELS[best_key], confidence

    # وإلا نستخدم Zero-shot كخطة بديلة على نص أطول (3000 حرف بدل 1500)
    result = zero_shot(text[:3000], list(CONTRACT_LABELS.values()),
                        hypothesis_template="هذا النص عبارة عن {}.")
    return result["labels"][0], result["scores"][0]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

LAW_FILE_NAMES = {"rental_law_reference.txt", "labor_law_reference.txt", "civil_commercial_reference.txt"}

def load_laws(laws_dir):
    law_chunks = []
    for file in sorted(Path(laws_dir).glob("*.txt")):
        if file.name not in LAW_FILE_NAMES:
            continue
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()
        articles = re.split(r"(المادة المرجعية\s*\d+[^:]*:)", text)
        current_title = None
        for part in articles:
            part = part.strip()
            if not part:
                continue
            if part.startswith("المادة المرجعية"):
                current_title = part
            elif current_title:
                law_chunks.append({"source": file.stem, "title": current_title, "text": part})
                current_title = None
    return law_chunks

law_chunks = load_laws(LAWS_DIR)
print(f"عدد المواد القانونية المرجعية: {len(law_chunks)}")

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
law_embeddings = embedder.encode([c["text"] for c in law_chunks], convert_to_numpy=True, show_progress_bar=True)

law_index = faiss.IndexFlatL2(law_embeddings.shape[1])
law_index.add(law_embeddings)

In [ ]:
def retrieve_relevant_law(clause_text, k=2):
    query_emb = embedder.encode([clause_text], convert_to_numpy=True)
    _, indices = law_index.search(query_emb, k)
    return [law_chunks[i] for i in indices[0]]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

In [ ]:
def build_risk_prompt(clause_title, clause_text, retrieved_laws):
    laws_context = "\n".join([f"- {l['title']} {l['text']}" for l in retrieved_laws])
    return f"""أنت مساعد قانوني. مهمتك تحليل بند من عقد ومقارنته بالقواعد القانونية التالية.

البند:
{clause_title}: {clause_text}

القواعد القانونية ذات الصلة:
{laws_context}

رد فقط بصيغة JSON بدون أي نص إضافي، بالشكل التالي بالضبط:
{{"is_risky": true or false, "risk_level": "low" or "medium" or "high", "reason": "شرح مختصر بالعربية", "suggested_rewrite": "صياغة بديلة أعدل للبند أو فارغة لو البند سليم"}}
"""

def parse_json_output(raw_text):
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

RISK_WEIGHTS = {"low": 5, "medium": 15, "high": 30}

In [ ]:
def analyze_full_contract(text, max_new_tokens=400, progress_callback=None):
    """يحلل نص عقد كامل: تصنيف النوع + كشف البنود الخطرة + درجة الخطورة.
    progress_callback: دالة اختيارية (fraction, message) لتحديث شريط التقدم في الواجهة."""
    contract_type, type_score = classify_contract_type(text)

    clauses = split_into_clauses(text)
    if not clauses:
        return {"error": "لم يتم التعرف على بنود بصيغة 'البند ...' في هذا الملف."}

    rows = []
    risky_count = 0
    total_score = 0

    for i, clause in enumerate(clauses):
        retrieved = retrieve_relevant_law(clause["text"], k=2)
        prompt = build_risk_prompt(clause["title"], clause["text"], retrieved)
        inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        outputs = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
        raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        parsed = parse_json_output(raw)

        if parsed and parsed.get("is_risky"):
            risky_count += 1
            level = parsed.get("risk_level", "low")
            total_score += RISK_WEIGHTS.get(level, 5)
            icon = "🔴" if level == "high" else "🟡" if level == "medium" else "🟢"
            reason = parsed.get("reason", "")
            rewrite = parsed.get("suggested_rewrite", "")
        else:
            level, icon, reason, rewrite = "-", "✅", "لا توجد مخالفة" if parsed else "تعذر تحليل رد النموذج", ""

        rows.append([clause["title"], icon, level, reason, rewrite])
        if progress_callback:
            progress_callback((i + 1) / len(clauses), f"تم تحليل البند {i+1} من {len(clauses)}")

    max_possible_score = len(clauses) * RISK_WEIGHTS["high"]
total_score = round((total_score / max_possible_score) * 100) if max_possible_score > 0 else 0
total_score = min(total_score, 100)
risk_label = "🔴 خطر مرتفع" if total_score >= 50 else "🟡 خطر متوسط" if total_score >= 20 else "🟢 خطر منخفض"

    return {
        "contract_type": contract_type,
        "type_score": type_score,
        "num_clauses": len(clauses),
        "risky_count": risky_count,
        "risk_score": total_score,
        "risk_label": risk_label,
        "rows": rows
    }

In [ ]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

def evaluate_contract(contract_name, rows):
    gt = ground_truth[contract_name]
    gt_risky = {c["clause_number"] for c in gt["risky_clauses"]}
    predicted_risky = {i for i, row in enumerate(rows, start=1) if row[1] != "✅"}

    tp = len(gt_risky & predicted_risky)
    fp = len(predicted_risky - gt_risky)
    fn = len(gt_risky - predicted_risky)

    if len(gt_risky) == 0:
        correct = 1.0 if len(predicted_risky) == 0 else 0.0
        return {"precision": correct, "recall": 1.0, "f1": correct, "tp": tp, "fp": fp, "fn": fn}

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

In [ ]:
all_metrics = {}
for name, text in contracts.items():
    result = analyze_full_contract(text)
    metrics = evaluate_contract(name, result["rows"])
    all_metrics[name] = metrics
    print(f"{name:25s} -> Precision={metrics['precision']:.2f}  Recall={metrics['recall']:.2f}  F1={metrics['f1']:.2f}")

avg_f1 = sum(m['f1'] for m in all_metrics.values()) / len(all_metrics)
print(f"\n📊 متوسط F1 على كل العقود: {avg_f1:.2f}")

In [ ]:
from pypdf import PdfReader
import docx

def extract_text_from_file(file_obj):
    file_path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)

    if file_path.lower().endswith(".pdf"):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    elif file_path.lower().endswith((".docx", ".doc")):
        doc = docx.Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

In [ ]:
import gradio as gr

def gradio_analyze(file_obj, progress=gr.Progress()):
    if file_obj is None:
        return "من فضلك ارفع ملف عقد أولاً.", None, None

    try:
        progress(0.05, desc="جاري استخراج النص من الملف...")
        text = extract_text_from_file(file_obj)

        if not text.strip():
            return "لم يتم العثور على نص داخل الملف.", None, None

        def cb(frac, msg):
            progress(0.1 + 0.85 * frac, desc=msg)

        progress(0.1, desc="جاري تصنيف نوع العقد وتحليل البنود...")
        result = analyze_full_contract(text, progress_callback=cb)

        if "error" in result:
            return result["error"], None, None

        summary = f"""
### نتيجة تحليل العقد

**نوع العقد المكتشف:** {result['contract_type']} (ثقة {result['type_score']:.0%})
**عدد البنود:** {result['num_clauses']}
**عدد البنود الخطرة:** {result['risky_count']}
**درجة الخطورة الكلية:** {result['risk_score']}/100 — {result['risk_label']}
"""
        df = pd.DataFrame(result["rows"], columns=["البند", "الحالة", "درجة الخطورة", "السبب", "صياغة بديلة مقترحة"])

        report_path = "/tmp/contract_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        return summary, df, report_path

    except Exception as e:
        return f"حدث خطأ أثناء المعالجة: {str(e)}", None, None

In [ ]:
custom_css = """
footer {visibility: hidden; display: none !important;}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="المساعد القانوني الذكي") as demo:
    gr.Markdown("""
    # ⚖️ المساعد القانوني الذكي لتحليل العقود
    ارفع عقد (**PDF**, **TXT**, أو **DOCX**) وسيتم تحليله تلقائيًا: تصنيف نوعه، كشف البنود المخالفة للقانون المصري،
    حساب درجة الخطورة، واقتراح صياغة بديلة للبنود الخطرة.
    """)

    with gr.Row():
        file_input = gr.File(label="ارفع ملف العقد (PDF, TXT, DOCX)", file_types=[".pdf", ".txt", ".docx", ".doc"])
        analyze_btn = gr.Button("🔍 حلل العقد", variant="primary")

    summary_output = gr.Markdown()
    table_output = gr.Dataframe(label="تحليل تفصيلي لكل بند", wrap=True)
    download_output = gr.File(label="تحميل التقرير كامل (JSON)")

    analyze_btn.click(fn=gradio_analyze, inputs=[file_input], outputs=[summary_output, table_output, download_output])

demo.launch(share=True, debug=True)